<a href="https://colab.research.google.com/github/DavidGGs5/Fundamentos/blob/main/Unidad2%20/%20sistema_prestamos_modular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Colab Notebooks/Sistema de prestamos"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/Sistema de prestamos


In [12]:

%%writefile datos.py
alumnos = ["26040200", "26040201", "26040202", "26040203", "26040204" ]
admins = ["123456", "234567"]
computadoras = [
 {"id": "PC-LC1-01", "aula": "LC1", "software": "Python, VS Code", "estado": "Disponible", "usuario": "", "fecha_reserva": "" },
{"id": "PC-LC1-02", "aula": "LC1", "software": "Python, Java, VS Code", "estado": "Disponible","usuario": "", "fecha_reserva": "" },
{"id": "PC-LC1-03", "aula": "LC1", "software": "AUTOCAD, MATLAB", "estado": "Disponible","usuario": "", "fecha_reserva": "" },
{"id": "PC-LC2-01", "aula": "LC2", "software": "Python, VS Code", "estado": "Disponible","usuario": "", "fecha_reserva": "" },
{"id": "PC-LC2-02", "aula": "LC2", "software": "Python, Java, VS Code", "estado": "Disponible","usuario": "", "fecha_reserva": "" },
{"id": "PC-LC2-03", "aula": "LC2", "software": "AUTOCAD, MATLAB", "estado": "Disponible","usuario": "", "fecha_reserva": "" },
{"id": "PC-LC3-01", "aula": "LC3", "software": "Python, VS Code", "estado": "Disponible","usuario": "", "fecha_reserva": "" },
{"id": "PC-LC3-02", "aula": "LC3", "software": "Python, Java, VS Code", "estado": "Disponible","usuario": "", "fecha_reserva": "" },
{"id": "PC-LC3-03", "aula": "LC3", "software": "AUTOCAD, MATLAB", "estado": "Disponible","usuario": "", "fecha_reserva": "" },
{"id": "PC-LC3-04", "aula": "LC3", "software": "Python, VS Code", "estado": "Mantenimiento","usuario": "", "fecha_reserva": "" }
]


Overwriting datos.py


In [13]:
%%writefile persistencia.py
import json
import os
from datos import alumnos, admins, computadoras

RUTA = '/content/drive/MyDrive/prestamos_datos.json'

def cargar_datos():
    if not os.path.exists(RUTA):
        return  # primera vez: se queda con los datos iniciales
    with open(RUTA, "r", encoding="utf-8") as f:
        datos = json.load(f)
    alumnos[:] = datos["alumnos"]
    admins[:] = datos["admins"]
    computadoras[:] = datos["computadoras"]
def guardar_datos():
    datos = {"alumnos": alumnos, "admins": admins, "computadoras": computadoras}
    with open(RUTA, "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=2)


Overwriting persistencia.py


In [14]:
%%writefile equipos.py
from datos import computadoras
from persistencia import guardar_datos



def consultar_computadoras():
  print("\n--- COMPUTADORAS ---")
  for i in (computadoras):
    print(f"| Aula: {i["aula"]} | ID: {i["id"]} | Software: {i["software"]} | Estado: {i["estado"]} | Usuario: {i["usuario"]} | Fecha de reservación: {i["fecha_reserva"]}")
    print("---------------------")

def buscar_soft():
    programa = input("\nIngrese el software que desea buscar: ")
    print("\n--- RESULTADOS ---")
    encontrado = False
    for i in computadoras:
        if programa.lower() in i["software"].lower():
            print(f"ID: {i['id']} | Aula: {i['aula']} | Software: {i['software']} | Estado: {i['estado']}")
            encontrado = True
    if encontrado == False:
        print("No se encontraron computadoras con ese software.")


def buscar_computadora_por_id(id_comp):
    for i in computadoras:
        if i["id"] == id_comp:
            return i
    return None


def buscar_pc_de_usuario(num_control):
    for pc in computadoras:
        if pc["usuario"] == num_control:
            return pc
    return None

def agreg_comp():
    print("\n--- AGREGAR COMPUTADORA ---")
    id_computadora = input("ID de la computadora: ")
    if buscar_computadora_por_id(id_computadora) is not None:
        print("Error: Ya existe una computadora con ese ID.")
        return
    aula = input("Aula: ")
    aulas = {pc["aula"] for pc in computadoras}
    if aula not in aulas:
        print("Esta aula no existe")
        return
    soow = input("Software instalado: ")
    nueva_pc = {
        "id": id_computadora,
        "aula": aula,
        "software": soow,
        "estado": "Disponible",
        "usuario": "",
        "fecha_reserva": ""
    }
    computadoras.append(nueva_pc)
    computadoras.sort(key=lambda i: (i.get("aula", ""), i.get("id", "")))
    guardar_datos()
    print("\nComputadora agregada correctamente.")

Overwriting equipos.py


In [15]:
%%writefile usuarios.py
from datos import alumnos, admins
from persistencia import guardar_datos



def agre_alu_o_admi():
  print("\n--- A quien vas a agregar ----")
  opcic = 0
  try:
    opcic = int(input("Presiona 1 para agregar alumno, "
                     "Presiona 2 si quieres agregar admin "
                     "Presiona 3 para cancelar: "))
  except ValueError:
    print("ingrese una opcion valida")



  if opcic == 1:
    alum = input("ingresa el numero de control del alumno: ")
    if alum in alumnos:
      print("Este alumno ya existe ")
      return
    alumnos.append(alum)
    print("El alumno a", alum , "a sido agregado con exito")
    guardar_datos()
    print(alumnos, "Actuales")
  elif opcic == 2:
    admi = input("Ingresa el numero de control del admin: ")
    if admi in admins:
      print("Este admin ya existe ")
      return
    admins.append(admi)
    print(f"El admin {admi} a sido agregado con exito")
    guardar_datos()
    print(admins, "Actuales")
  elif opcic == 3:
    return

Overwriting usuarios.py


In [16]:
%%writefile prestamos.py
from datetime import datetime, timedelta
from datos import alumnos, admins, computadoras
from persistencia import guardar_datos
from equipos import buscar_computadora_por_id, buscar_pc_de_usuario




def solic_pres(num_control):
    pc_actual = buscar_pc_de_usuario(num_control)
    if pc_actual is not None and num_control in alumnos:
        print(f"Ya tienes asignada la {pc_actual['id']}")
        return
    id_computadora = input("Ingrese el ID de la computadora: ")
    i = buscar_computadora_por_id(id_computadora)
    if i is None:
      print("La computadora no existe.")
      return
    elif i["estado"] == "Disponible":
      i["estado"] = "Prestada"
      i["usuario"] = num_control
      print("Préstamo registrado correctamente a favor del alumno.", num_control)
      print(f"Computadora: {i['id']} | Aula: {i['aula']}")
      guardar_datos()
      return
    elif i["estado"] != "Disponible":
      print(f"La computadora no está disponible (Estado actual: {i['estado']}).")
      return



def devo_comp(num_control):
    print("\n--- DEVOLVER COMPUTADORA ---")
    id_computadora = input("Ingrese el ID de la computadora: ")
    i = buscar_computadora_por_id(id_computadora)
    if i is None:
        print("La computadora no existe.")
    elif i["estado"] == "Prestada" or i["estado"] == "Reservada":
      if i["usuario"] == num_control:
        i["estado"] = "Disponible"
        i["usuario"]= ""
        i["fecha_reserva"] = ""
        print("Computadora devuelta correctamente.")
        print(f"Computadora: {i['id']} | Aula: {i['aula']}")
        guardar_datos()
        return
      else:
        print("Esta computadora no fue prestada a este usuario")
    else:
        print("La computadora no esta registrada como prestada. Estado:", i["estado"])


def crea_reserv(num_control):
  usuario = buscar_pc_de_usuario(num_control)
  if usuario is not None and num_control in alumnos:
    print("Este usuario ya tiene una computadora prestada o reservada")
    print(f"Ya tienes asignada la {usuario["id"]}")
    return
  else:
    print("\n--- CREAR RESERVACIÓN ---")
    hoy = datetime.now().date()
    limite_maximo = hoy + timedelta(days=3)
    print("Hoy es: ", hoy)
    print("")
    print("Puedes reservar hasta: ", limite_maximo)
    print("")
    id_computadora = input("Ingrese el ID de la computadora: ")
    i = buscar_computadora_por_id(id_computadora)
    if i is None:
      print("La compu no fue encontrada")
      return
    if i["estado"] != "Disponible":
      print("Esta computadora actualmente no esta disponible")
      return

      fecha_solicitada = input("Ingresa la fecha que quieres reservar, usando el formato (DD/MM/YYYY): ")
      try:
        fecha_reservacion = datetime.strptime(fecha_solicitada, "%d/%m/%Y").date()
      except ValueError:
            print("Error: El formato de fecha no es válido. Debe ser DD/MM/YYYY.")
            return

      if hoy <= fecha_reservacion <= limite_maximo:
        i["estado"] = "Reservada"
        i["usuario"] = num_control
        i["fecha_reserva"] = fecha_reservacion.strftime("%d/%m/%Y")
        print("\n==============================================")
        print(" ¡Reservación registrada con éxito!")
        print(f" Computadora: {i['id']} (Aula: {i['aula']})")
        print(f" Reservada para el alumno: {num_control}")
        print(f" Fecha programada: {i['fecha_reserva']}")
        print("==============================================")
        guardar_datos()
        return
      elif fecha_reservacion < hoy:
          print("No puedes solicitar la computadora para fechas pasadas.")
          return
      elif fecha_reservacion > limite_maximo:
          print("La fecha excede el límite permitido:", limite_maximo)
          return


def liberar_reservas_vencidas():
    hoy = datetime.now().date()
    for i in computadoras:
        if i["estado"] == "Reservada" and i["fecha_reserva"]:
            fecha = datetime.strptime(i["fecha_reserva"], "%d/%m/%Y").date()
            if fecha < hoy:
                i["estado"] = "Disponible"
                i["usuario"] = ""
                i["fecha_reserva"] = ""
                guardar_datos()

Overwriting prestamos.py


In [17]:
%%writefile menu.py
from datos import alumnos, admins
from equipos import consultar_computadoras, buscar_soft, agreg_comp
from usuarios import agre_alu_o_admi
from prestamos import solic_pres, devo_comp, crea_reserv



def sistema_de_prestamo (num_control):
  opcion = 0
  while True:
    print("\n==============================")
    print("      Sistema de prestamos  ")
    print("================================")
    print("1. Consultar computadoras y estados")
    print("2. Buscar por software")
    print("3. Solicitar prestamo")
    print("4. Devolver computadora")
    print("5. Crear reservacion")
    print("6. Agregar computadora")
    print("7. Agregar alumno o admin")
    print("8. Cambiar de cuenta")
    print("9. Salir")
    try:
      opcion = int(input("Seleccione una opcion: "))
    except ValueError:
      print("Error, debe ingresar un numero correspondiente al menu")
      continue

    if opcion == 1:
      consultar_computadoras()
    elif opcion == 2:
      buscar_soft()
    elif opcion == 3:
      solic_pres(num_control)
    elif opcion == 4:
      devo_comp(num_control)
    elif opcion == 5:
     crea_reserv(num_control)
    elif opcion == 6:
      agreg_comp()
    elif opcion == 7:
      agre_alu_o_admi()
    elif opcion == 8:
      nuevo_control = input("Ingresa tu numero de control: ")
      if nuevo_control in alumnos or nuevo_control in admins:
        num_control = nuevo_control
      else:
        print("El alumno o admi no existe")

    elif opcion == 9:
      print(f"Saliendo del sistema... Hasta luego {num_control}")
      break
    else:
      print("Opcion no valida")

Overwriting menu.py


In [18]:
%%writefile main.py
from datos import alumnos, admins
from persistencia import cargar_datos
from prestamos import liberar_reservas_vencidas
from menu import sistema_de_prestamo



def main():

  cargar_datos()
  liberar_reservas_vencidas()


  num_control = input("Ingresa tu numero de control: ")
  if num_control in alumnos or num_control in admins:
      sistema_de_prestamo(num_control)
  else:
      print("El alumno o admi no existe")

if __name__ == "__main__":
    main()

Overwriting main.py


In [19]:
from main import main
main()

Ingresa tu numero de control: 26040200

      Sistema de prestamos  
1. Consultar computadoras y estados
2. Buscar por software
3. Solicitar prestamo
4. Devolver computadora
5. Crear reservacion
6. Agregar computadora
7. Agregar alumno o admin
8. Cambiar de cuenta
9. Salir
Seleccione una opcion: 1

--- COMPUTADORAS ---
| Aula: LC1 | ID: PC-LC1-01 | Software: Python, VS Code | Estado: Disponible | Usuario:  | Fecha de reservación: 
---------------------
| Aula: LC1 | ID: PC-LC1-02 | Software: Python, Java, VS Code | Estado: Disponible | Usuario:  | Fecha de reservación: 
---------------------
| Aula: LC1 | ID: PC-LC1-03 | Software: AUTOCAD, MATLAB | Estado: Disponible | Usuario:  | Fecha de reservación: 
---------------------
| Aula: LC2 | ID: PC-LC2-01 | Software: Python, VS Code | Estado: Disponible | Usuario:  | Fecha de reservación: 
---------------------
| Aula: LC2 | ID: PC-LC2-02 | Software: Python, Java, VS Code | Estado: Disponible | Usuario:  | Fecha de reservación: 
---------